# An Explainable Semi-Supervised deep learning framework for mineral prospectivity mapping


## **Preparation: Configuring the Environment**

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!apt-get install -y --no-install-recommends --no-install-suggests \
    gdal-bin libgdal-dev libnetcdf-dev libglib2.0-dev

In [ ]:
!wget https://github.com/GenericMappingTools/gmt/releases/download/6.5.0/gmt-6.5.0-src.tar.xz
!tar -xvf gmt-6.5.0-src.tar.xz
!cd gmt-6.5.0 && mkdir build && cd build && cmake .. && make -j4 && make install

In [ ]:
!pip install -r "/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/requirements.txt"

## **Importing Required Libraries**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import contextily as cx
import geopandas as gpd
from ipywidgets import interact
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import math
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
from numpy import genfromtxt
from osgeo import gdal
from osgeo import osr
import os
import pandas as pd
import pickle
import rioxarray as rxr
from shapely.geometry import Point
import shapely.strtree
from skimage import exposure, util
from skimage.feature import graycomatrix, graycoprops
from tqdm.notebook import tqdm
import xarray as xr
import csv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
import pygmt
from scipy.spatial import cKDTree
from shapely.geometry import MultiLineString, LineString
from geopy import distance
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.interpolate import griddata
from scipy.interpolate import RectBivariateSpline
from matplotlib.colors import Normalize
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Nadam
from tensorflow.keras import backend as K
import random
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint
import shap
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix
from sklearn.metrics import accuracy_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score, average_precision_score

## **Data Preparation and Processing**

### REE Mineral Occurrences

In [ ]:
# Read the data file of REE deposit distribution
min_occ = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/GIS/ree.shp')
# Read the data file of study area border
frame = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/GIS/Clip_frame.shp')
# Get the minimum and maximum coordinates of the border to define the map range displayed
bounds = frame.bounds
extent = [bounds.loc[0]['minx'], bounds.loc[0]['maxx'], bounds.loc[0]['miny'], bounds.loc[0]['maxy']]
# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(10, 10))
min_occ.plot(ax=ax, edgecolor='black', color='yellow')
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.set_title('REE Mineral Occurrences')
plt.show()

### Geophysical data


#### Magnetics

In [ ]:
# Read magnetic data files, with each path pointing to a different magnetic data raster file
magnetic_files = [
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_1VD.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_AGC.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_AS.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_BigE.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_BigT.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_CAUCHY_3rd.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_PseudoGrav.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_Tilt.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_TZ.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Magnetics/SA_TMI_VRTP_Tzz.tif'
]

# Use the @interact decorator to create an interactive control that selects a file from magnetic data files for display
@interact(file=magnetic_files)
def show_dist(file):
    raster = rxr.open_rasterio(file, masked=True).squeeze()

    # If the coordinate reference frame (CRS) of the raster data is different from the CRS of the border of the study area,
    # it is reprojected to the CRS of the border
    if raster.rio.crs.to_epsg() != frame.crs.to_epsg():
        raster = raster.rio.reproject(frame.crs)

    # Convert raster data to a NumPy array
    raster_array = raster.values

    # Calculate the mean and standard deviation of the raster data to adjust the color range displayed in the image
    v_mean = np.nanmean(raster_array)
    v_std = np.nanstd(raster_array)

    # Create a drawing object and display it
    fig, ax = plt.subplots(figsize=(10, 10))
    frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
    cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
    cb = ax.imshow(raster_array, cmap='Spectral_r', extent=extent, vmin=v_mean-v_std, vmax=v_mean+v_std)
    divider = make_axes_locatable(ax)
    cax = divider.new_vertical(size='5%', pad=0.5, pack_start=True)
    fig.add_axes(cax)
    filename = os.path.splitext(os.path.basename(file))[0]
    plt.colorbar(cb, orientation='horizontal', label=filename, cax=cax)
    plt.show()

#### Gravity

In [ ]:
# Read gravity data files, with each path pointing to a different gravity data raster file
gravity_files = [
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Gravity/SA_GRAV_1VD_ONSHORE.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Gravity/SA_GRAV_UC1000_RESIDUAL_ONSHORE.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Gravity/SA_GRAV_ONSHORE.tif'
]

# Use the @interact decorator to create an interactive control that selects a file from gravity data files for display
@interact(file=gravity_files)
def show_dist(file):
    raster = rxr.open_rasterio(file, masked=True).squeeze()

    # If the coordinate reference frame (CRS) of the raster data is different from the CRS of the border of the study area,
    # it is reprojected to the CRS of the border
    if raster.rio.crs.to_epsg() != frame.crs.to_epsg():
        raster = raster.rio.reproject(frame.crs)

    # Convert raster data to a NumPy array
    raster_array = raster.values

    # Calculate the mean and standard deviation of the raster data to adjust the color range displayed in the image
    v_mean = np.nanmean(raster_array)
    v_std = np.nanstd(raster_array)

    # Create a drawing object and display it
    fig, ax = plt.subplots(figsize=(10, 10))
    frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
    cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
    cb = ax.imshow(raster_array, cmap='Spectral_r', extent=extent, vmin=v_mean-v_std, vmax=v_mean+v_std)
    divider = make_axes_locatable(ax)
    cax = divider.new_vertical(size='5%', pad=0.5, pack_start=True)
    fig.add_axes(cax)
    filename = os.path.splitext(os.path.basename(file))[0]
    plt.colorbar(cb, orientation='horizontal', label=filename, cax=cax)
    plt.show()


#### Radiometrics

In [ ]:
# Read radiometric data files, with each path pointing to a different radiometric data raster file
radiometric_files = [
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Radiometrics/SA_RAD_DOSE.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Radiometrics/SA_RAD_K.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Radiometrics/SA_RAD_TH.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Radiometrics/SA_RAD_U.tif'
]

# Use the @interact decorator to create an interactive control that selects a file from Radiometrics data files for display
@interact(file=radiometric_files)
def show_dist(file):
    raster = rxr.open_rasterio(file, masked=True).squeeze()

    # If the coordinate reference frame (CRS) of the raster data is different from the CRS of the border of the study area,
    # it is reprojected to the CRS of the border
    if raster.rio.crs.to_epsg() != frame.crs.to_epsg():
        raster = raster.rio.reproject(frame.crs)

    # Convert raster data to a NumPy array
    raster_array = raster.values

    # Calculate the mean and standard deviation of the raster data to adjust the color range displayed in the image
    v_mean = np.nanmean(raster_array)
    v_std = np.nanstd(raster_array)

    # Create a drawing object and display it
    fig, ax = plt.subplots(figsize=(10, 10))
    frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
    cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
    cb = ax.imshow(raster_array, cmap='Spectral_r', extent=extent, vmin=v_mean-v_std, vmax=v_mean+v_std)
    divider = make_axes_locatable(ax)
    cax = divider.new_vertical(size='5%', pad=0.5, pack_start=True)
    fig.add_axes(cax)
    filename = os.path.splitext(os.path.basename(file))[0]
    plt.colorbar(cb, orientation='horizontal', label=filename, cax=cax)
    plt.show()

#### Remote Sensing

In [ ]:
# Read remote sensing data files, with each path pointing to a different remote sensing data raster file
remote_sensing_files = [
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/AlOH_Group_Composition.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/AlOH_Group_Content.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/FeOH_Group_Content.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Ferric_Oxide_Composition.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Ferric_Oxide_Content.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Ferrous_Iron_Content_in_MgOH.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Ferrous_Iron_Index.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Green_Vegetation.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Gypsum_Index.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Kaolin_Group_Index.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/MgOH_Group_Composition.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/MgOH_Group_Content.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Opaque_Index.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Quartz_Index.tif',
    '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Remote Sensing/Silica_Index.tif'
]

# Use the @interact decorator to create an interactive control that selects a file from Remote Sensing data files for display
@interact(file=remote_sensing_files)
def show_dist(file):
    raster = rxr.open_rasterio(file, masked=True).squeeze()

    # If the coordinate reference frame (CRS) of the raster data is different from the CRS of the border of the study area,
    # it is reprojected to the CRS of the border
    if raster.rio.crs.to_epsg() != frame.crs.to_epsg():
        raster = raster.rio.reproject(frame.crs)

    # Convert raster data to a NumPy array
    raster_array = raster.values

    # Calculate the mean and standard deviation of the raster data to adjust the color range displayed in the image
    v_mean = np.nanmean(raster_array)
    v_std = np.nanstd(raster_array)

    # Create a drawing object and display it
    fig, ax = plt.subplots(figsize=(10, 10))
    frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
    cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
    cb = ax.imshow(raster_array, cmap='Spectral_r', extent=extent, vmin=v_mean-v_std, vmax=v_mean+v_std)
    divider = make_axes_locatable(ax)
    cax = divider.new_vertical(size='5%', pad=0.5, pack_start=True)
    fig.add_axes(cax)
    filename = os.path.splitext(os.path.basename(file))[0]
    plt.colorbar(cb, orientation='horizontal', label=filename, cax=cax)
    plt.show()

#### DEM

In [ ]:
# Read the DEM data file
dem_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/DEM/DEM_9s.tif'
raster = rxr.open_rasterio(dem_file, masked=True).squeeze()

# If the coordinate reference frame (CRS) of the raster data is different from the CRS of the border of the study area,
# it is reprojected to the CRS of the border
if raster.rio.crs.to_epsg() != frame.crs.to_epsg():
    raster = raster.rio.reproject(frame.crs)

# Convert raster data to a NumPy array
raster_array = raster.values

# Calculate the mean and standard deviation of the raster data to adjust the color range displayed in the image
v_mean = np.nanmean(raster_array)
v_std = np.nanstd(raster_array)

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(10, 10))
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
cb = ax.imshow(raster_array, cmap='Spectral_r', extent=extent, vmin=v_mean-v_std, vmax=v_mean+v_std)
divider = make_axes_locatable(ax)
cax = divider.new_vertical(size='5%', pad=0.5, pack_start=True)
fig.add_axes(cax)
plt.colorbar(cb, orientation='horizontal', label='Elevation (m)', cax=cax)
plt.show()

## Geological factors

#### Archean_Early_Mesoproterozoic_Faults

In [ ]:
# Read the fault data file
faults = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Geology factors/original/Archean_Early_Mesoproterozoic_Faults.shp')

# Transform the coordinate reference system of fault data
faults = faults.to_crs('epsg:4283')

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
faults.plot(ax=ax, color=None, edgecolor='black', linewidth=1)
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=1)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.set_title('Faults')
plt.show()

In [ ]:
# Read the faults data file and the frame data file
# Transform the coordinate reference system
faults = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Geology factors/original/Archean_Early_Mesoproterozoic_Faults.shp')
faults = faults.to_crs('epsg:4283')
frame = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/GIS/Clip_frame.shp')
frame = frame.to_crs('epsg:4283')

# Get the border of the study area
bounds = frame.total_bounds
xmin, ymin, xmax, ymax = bounds

# Set the cell size of the grid and generate grid point coordinates covering the study area
cell_size = 0.001
x = np.arange(xmin, xmax, cell_size)
y = np.arange(ymin, ymax, cell_size)
x_grid, y_grid = np.meshgrid(x, y)
points = np.vstack([x_grid.ravel(), y_grid.ravel()]).T

# Set the coordinate reference system of the grid points
# and filter the grid points located in the study area
points_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(points[:, 0], points[:, 1]), crs='epsg:4283')
points_within_frame_faults = points_gdf[points_gdf.within(frame.unary_union)]

#Calculate the distance from each grid point to the nearest fault border
distances = points_within_frame_faults.geometry.apply(lambda point: faults.distance(point).min())

# Normalize the distance data
# and invert it so that points close to the fault show higher values
min_distance = distances.min()
max_distance = distances.max()
normalized_distances = (distances - min_distance) / (max_distance - min_distance)
reversed_distances = 1 - normalized_distances
points_within_frame_faults['distance'] = reversed_distances

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
faults.plot(ax=ax, color=None, edgecolor='black', linewidth=1)
points_plot = points_within_frame_faults.plot(ax=ax, column='distance', cmap='viridis', markersize=1, alpha=0.6)
frame.boundary.plot(ax=ax, edgecolor='red', linewidth=1)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
cbar = plt.colorbar(points_plot.collections[0], ax=ax, orientation="vertical")
cbar.set_label('Reversed Distance')
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.set_title('Fault Distance Distribution')
plt.show()


#### Felsic granite

In [ ]:
# Read the granite data file
granite = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Geology factors/original/Felsic_granite.shp')

# Transform the coordinate reference system of granite data
granite = granite.to_crs('epsg:4283')

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
granite.plot(ax=ax, color=None, edgecolor='black', linewidth=1)
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=1)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.set_title('Felsic granite')
plt.show()

In [ ]:
# Read the granite data file and the frame data file
# Transform the coordinate reference system
granite = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Geology factors/original/Felsic_granite.shp')
granite = granite.to_crs('epsg:4283')
frame = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/GIS/Clip_frame.shp')
frame = frame.to_crs('epsg:4283')

# Get the border of the study area
bounds = frame.total_bounds
xmin, ymin, xmax, ymax = bounds

# Set the cell size of the grid and generate grid point coordinates covering the study area
cell_size = 0.001
x = np.arange(xmin, xmax, cell_size)
y = np.arange(ymin, ymax, cell_size)
x_grid, y_grid = np.meshgrid(x, y)
points = np.vstack([x_grid.ravel(), y_grid.ravel()]).T

# Set the coordinate reference system of the grid points
# and filter the grid points located in the study area
points_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(points[:, 0], points[:, 1]), crs='epsg:4283')
points_within_frame_granite = points_gdf[points_gdf.within(frame.unary_union)]

# Calculate the distance from each grid point to the nearest granite border
distances = points_within_frame_granite.geometry.apply(lambda point: granite.distance(point).min())

# Normalize the distance data
# and invert it so that points close to the fault show higher values
min_distance = distances.min()
max_distance = distances.max()
normalized_distances = (distances - min_distance) / (max_distance - min_distance)
reversed_distances = 1 - normalized_distances
points_within_frame_granite['distance'] = reversed_distances

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
granite.plot(ax=ax, color=None, edgecolor='black', linewidth=1)
points_plot = points_within_frame_granite.plot(ax=ax, column='distance', cmap='viridis', markersize=1, alpha=0.6)
frame.boundary.plot(ax=ax, edgecolor='red', linewidth=1)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
cbar = plt.colorbar(points_plot.collections[0], ax=ax, orientation="vertical")
cbar.set_label('Reversed Distance')
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.set_title('Felsic Granite Distance Distribution')
plt.show()


#### Mesoproterozoic strata

In [ ]:
# Read the strata data file
strata = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Geology factors/original/Mesoproterozoic.shp')

# Transform the coordinate reference system of granite data
strata = strata.to_crs('epsg:4283')

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
strata.plot(ax=ax, color=None, edgecolor='black', linewidth=1)
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=1)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.set_title('Mesoproterozoic strata')
plt.show()

In [ ]:
# Read the strata data file and the frame data file
# Transform the coordinate reference system
strata = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/Geology factors/original/Mesoproterozoic.shp')
strata = strata.to_crs('epsg:4283')
frame = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/GIS/Clip_frame.shp')
frame = frame.to_crs('epsg:4283')

# Get the border of the study area
bounds = frame.total_bounds
xmin, ymin, xmax, ymax = bounds

# Set the cell size of the grid and generate grid point coordinates covering the study area
cell_size = 0.001
x = np.arange(xmin, xmax, cell_size)
y = np.arange(ymin, ymax, cell_size)
x_grid, y_grid = np.meshgrid(x, y)
points = np.vstack([x_grid.ravel(), y_grid.ravel()]).T

# Set the coordinate reference system of the grid points
# and filter the grid points located in the study area
points_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy(points[:, 0], points[:, 1]), crs='epsg:4283')
points_within_frame_strata = points_gdf[points_gdf.within(frame.unary_union)]

# Calculate the distance from each grid point to the nearest strata border
distances = points_within_frame_strata.geometry.apply(lambda point: strata.distance(point).min())

# Normalize the distance data
# and invert it so that points close to the fault show higher values
min_distance = distances.min()
max_distance = distances.max()
normalized_distances = (distances - min_distance) / (max_distance - min_distance)
reversed_distances = 1 - normalized_distances
points_within_frame_strata['distance'] = reversed_distances

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
strata.plot(ax=ax, color=None, edgecolor='black', linewidth=1)
points_plot = points_within_frame_strata.plot(ax=ax, column='distance', cmap='viridis', markersize=1, alpha=0.6)
frame.boundary.plot(ax=ax, edgecolor='red', linewidth=1)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
cbar = plt.colorbar(points_plot.collections[0], ax=ax, orientation="vertical")
cbar.set_label('Reversed Distance')
ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax.set_title('Mesoproterozoic Strata Distance Distribution')
plt.show()


### Extract the Coordinates of Mineral Occurrences

In [ ]:
# Export the coordinates of REE deposits to a CSV file
deposit_coords_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/data_deposit_coords_extend_109.csv'

# If file exists
if os.path.isfile(deposit_coords_file):
    print('The coordinates of deposits already exist.')
    deposit_coords = pd.read_csv(deposit_coords_file, index_col=False)
    # Obtain the number of deposits
    deposit_num = int(deposit_coords.shape[0])
    # Extract X and Y coordinates of deposits
    deposit_x = pd.Series.tolist(deposit_coords['X'])
    deposit_y = pd.Series.tolist(deposit_coords['Y'])

# If the file does not exist
else:
    # Extract the X and Y coordinates of deposits from the min_occ data
    # as well as the number of deposits
    # and add a label column.
    deposit_x = min_occ.geometry.x
    deposit_y = min_occ.geometry.y
    deposit_num = min_occ.shape[0]
    deposit_coords = pd.DataFrame(deposit_x, columns=['X'])
    deposit_coords['Y'] = deposit_y
    deposit_coords['label'] = 1
    deposit_coords.to_csv(deposit_coords_file, index=False)
    print(f'The coordinates of deposits have been saved to {deposit_coords_file}.')

#### Geophysical features

In [ ]:
# Calculate mean and standard deviation in a buffer zone (circle) surrounding each target point
def get_mean_std(xs, ys, raster_np, bounds, radius):
    points = [Point(x, y) for x, y in zip(xs, ys)]
    means = []
    stds = []

    for point in points:
        x = point.x
        y = point.y
        xx = []
        yy = []

        # Check if the target point is outside the study area
        if x < bounds[0] or x > bounds[2] or y < bounds[1] or y > bounds[3]:
            means.append(np.nan)
            stds.append(np.nan)
        else:
            # Calculate the coordinates of the target point in the raster image
            x_origin = math.floor((x-bounds[0])/(bounds[2]-bounds[0])*(raster_np.shape[1]-1))
            y_origin = math.floor((y-bounds[1])/(bounds[3]-bounds[1])*(raster_np.shape[0]-1))

            # Calculate the coordinates of the buffer zone (circle) in the raster image
            radius_= math.ceil(radius*raster_np.shape[1]/(bounds[2]-bounds[0]))
            points_circle = []

            # Generate the coordinates of the points in the buffer zone
            for xr in range(-radius_, radius_+1):
                Y = int((radius_*radius_-xr*xr)**0.5)
                for yr in range(-Y, Y+1):
                    xc = xr + x_origin
                    yc = yr + y_origin
                    if xc >= 0 and xc <= raster_np.shape[1]-1 and yc >= 0 and yc <= raster_np.shape[0]-1:
                        points_circle.append((xc, yc))

            # Extracts the raster values in the buffer zone (circle)
            for p in points_circle:
                xx.append(p[0])
                yy.append(raster_np.shape[0]-1-p[1])

            # Calculate and store the mean and standard deviation in the buffer zone (circle)
            means.append(np.nanmean(raster_np[yy, xx]))
            stds.append(np.nanstd(raster_np[yy, xx]))

    return means, stds

# Calculate dissimilarity and correlation in a window (square) surrounding each target point
def get_diss_corr(xs, ys, raster_np, bounds, patch_size):
    points = [Point(x, y) for x, y in zip(xs, ys)]
    dissimilarity = []
    correlation = []

    for point in points:
        x = point.x
        y = point.y

        # Calculate the border of the window (square) in the raster image
        left = max(x-patch_size/2, bounds[0])
        right = min(x+patch_size/2, bounds[2])
        top = min(y+patch_size/2, bounds[3])
        bottom = max(y-patch_size/2, bounds[1])

        # Converts window border to indexes in the raster image
        left_idx = math.floor((left-bounds[0])/(bounds[2]-bounds[0])*(raster_np.shape[1]-1))
        right_idx = math.floor((right-bounds[0])/(bounds[2]-bounds[0])*(raster_np.shape[1]-1))
        bottom_idx = math.floor((bottom-bounds[1])/(bounds[3]-bounds[1])*(raster_np.shape[0]-1))
        top_idx = math.floor((top-bounds[1])/(bounds[3]-bounds[1])*(raster_np.shape[0]-1))

        # Generate the coordinates of the points in the window
        xs = np.arange(left_idx, right_idx+1)
        ys = np.arange(raster_np.shape[0]-1-top_idx, raster_np.shape[0]-1-bottom_idx+1)
        xm, ym = np.meshgrid(xs, ys)

        if np.isnan(raster_np[ym, xm]).all():
            dissimilarity.append(np.nan)
            correlation.append(np.nan)
        else:
            # Normalize the raster values in the window
            # and calculate Gray level co-occurrence matrix
            raster_scaled = exposure.rescale_intensity(raster_np[ym, xm], in_range=(np.nanmin(raster_np[ym, xm]), np.nanmax(raster_np[ym, xm])), out_range=(0, 1))
            raster_ubyte = util.img_as_ubyte(raster_scaled)
            glcm = graycomatrix(raster_ubyte, distances=[5], angles=[0], levels=256, symmetric=True, normed=True)
            dissimilarity.append(graycoprops(glcm, 'dissimilarity')[0, 0])
            correlation.append(graycoprops(glcm, 'correlation')[0, 0])

    return dissimilarity, correlation

# Concatenate and export the features generated from the functions above
def get_grid_data(xs, ys, grid_filenames):
    grid_features = []
    grid_column_names = []

    for grid in tqdm(grid_filenames):
        prefix = os.path.splitext(os.path.basename(grid))[0]
        grid_column_names.append(prefix+'_mean')
        grid_column_names.append(prefix+'_std')
        grid_column_names.append(prefix+'_dissimilarity')
        grid_column_names.append(prefix+'_correlation')

        raster = rxr.open_rasterio(grid, masked=True).squeeze()

        # reproject if required
        if raster.rio.crs != frame.crs:
            raster = raster.rio.reproject(frame.crs)

        bounds = (raster.rio.bounds())
        raster_np = raster.values

        # Calculate the mean and standard deviation and add them to the feature list
        means, stds = get_mean_std(xs, ys, raster_np, bounds, 0.011548)
        grid_features.append(means)
        grid_features.append(stds)

        # Calculate the dissimilarity and correlation and add them to the feature list
        dissimilarity,  correlation = get_diss_corr(xs, ys, raster_np, bounds, 0.011548)
        grid_features.append(dissimilarity)
        grid_features.append(correlation)

        del raster
        del raster_np

    return pd.DataFrame(np.array(grid_features).T, columns=grid_column_names)

# Merge all raster file lists
grid_filenames = magnetic_files + gravity_files + radiometric_files + remote_sensing_files

# Export the grid feature of REE deposits to a CSV file
deposit_grid_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/deposit_RRMG_feature_d0.0115_all_109.csv'

# If file exists
if os.path.isfile(deposit_grid_file):
    print('The grid dataset (deposits) already exists.')
    deposit_grid_data = pd.read_csv(deposit_grid_file, index_col=False)

# If file does not exist
else:
    # Calculate the grid features and save them to a CSV file
    deposit_grid_data = get_grid_data(deposit_x, deposit_y, grid_filenames)
    deposit_grid_data = deposit_grid_data.dropna(axis=1, thresh=round(deposit_num*0.9))
    deposit_grid_data.to_csv(deposit_grid_file, index=False)
    print(f'The grid dataset (deposits) has been saved to {deposit_grid_file}.')

In [ ]:
# Calculate mean in a buffer zone (circle) surrounding each target point
def get_mean(xs, ys, raster_grad, bounds, radius):
    points = [Point(x, y) for x, y in zip(xs, ys)]
    means = []

    for point in points:
        x = point.x
        y = point.y
        xx = []
        yy = []

        # Check if the target point is outside the study area
        if x < bounds[0] or x > bounds[2] or y < bounds[1] or y > bounds[3]:
            means.append(np.nan)
        else:
            # Calculate the coordinates of the target point in the raster image
            x_origin = math.floor((x-bounds[0])/(bounds[2]-bounds[0])*(raster_grad.shape[1]-1))
            y_origin = math.floor((y-bounds[1])/(bounds[3]-bounds[1])*(raster_grad.shape[0]-1))

            # Calculate the coordinates of the buffer zone (circle) in the raster image
            radius_ = math.ceil(radius*raster_grad.shape[1]/(bounds[2]-bounds[0]))
            points_circle = []

            # Generate the coordinates of the points in the buffer zone
            for xr in range(-radius_, radius_+1):
                Y = int((radius_*radius_-xr*xr)**0.5)
                for yr in range(-Y, Y+1):
                    xc = xr + x_origin
                    yc = yr + y_origin
                    if xc >= 0 and xc <= raster_grad.shape[1]-1 and yc >= 0 and yc <= raster_grad.shape[0]-1:
                        points_circle.append((xc, yc))

            # Extracts the raster values in the buffer zone (circle)
            for p in points_circle:
                xx.append(p[0])
                yy.append(raster_grad.shape[0]-1-p[1])

            # Calculate and store the mean in the buffer zone (circle)
            means.append(np.nanmean(raster_grad[yy, xx]))

    return means

# Calculate gradient and mean of target point
def get_gradient_data(xs, ys, elevation_files):
    grid_features = []
    grid_column_names = []

    for grid in tqdm(elevation_files):
        prefix = os.path.splitext(os.path.basename(grid))[0]
        grid_column_names.append(prefix+'_dx')
        grid_column_names.append(prefix+'_dy')

        raster = rxr.open_rasterio(grid, masked=True).squeeze()

        bounds = (raster.rio.bounds())
        raster_np = np.array(raster)

        # Caculate gradient of X and Y direction
        raster_grad_x = np.gradient(raster_np)[1]
        raster_grad_y = np.gradient(raster_np)[0]

        # Calculate the mean in the gradient direction
        means_x = get_mean(xs, ys, raster_grad_x, bounds, 0.011548)
        means_y = get_mean(xs, ys, raster_grad_y, bounds, 0.011548)
        grid_features.append(means_x)
        grid_features.append(means_y)

        del raster
        del raster_np

    return pd.DataFrame(np.array(grid_features).T, columns=grid_column_names)

# Export the elevation feature of REE deposits to a CSV file
deposit_elev_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/deposit_dem_d0.0115_all_109.csv'

# If file exists
if os.path.isfile(deposit_elev_file):
    print('The elevation dataset (deposits) already exists.')
    deposit_elev_data = pd.read_csv(deposit_elev_file, index_col=False)

# If file does not exist
else:
    # Calculate the elevation features and save them to a CSV file
    deposit_elev_data = get_gradient_data(deposit_x, deposit_y, [dem_file])
    deposit_elev_data = deposit_elev_data.dropna(axis=1, thresh=round(deposit_num*0.9))
    deposit_elev_data.to_csv(deposit_elev_file, index=False)
    print(f'The elevation dataset (deposits) has been saved to {deposit_elev_file}.')

### Geological feature

In [ ]:
# Read the REE deposit data file
# Transform the coordinate reference system
deposit_coords = pd.read_csv(deposit_coords_file, index_col=False)
deposit_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(deposit_coords['X'], deposit_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the deposit points with the fault grid points in the frame
deposit_points_join = gpd.sjoin_nearest(deposit_points, points_within_frame_faults, how='left', distance_col='distance_to_grid')
deposit_points_join['fault_distance'] = deposit_points_join['distance']

# Extract the deposit point coordinates and the distance from deposit points to faults
# and save them to a CSV file
result_deposit_fault = deposit_points_join[['geometry', 'fault_distance']]
result_deposit_fault.to_csv('deposit_fault_distances.csv', index=False)

In [ ]:
# Read the REE deposit data file
# Transform the coordinate reference system
deposit_coords = pd.read_csv(deposit_coords_file, index_col=False)
deposit_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(deposit_coords['X'], deposit_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the deposit points with the granite grid points in the frame
deposit_points_join = gpd.sjoin_nearest(deposit_points, points_within_frame_granite, how='left', distance_col='distance_to_grid')
deposit_points_join['granite_distance'] = deposit_points_join['distance']

# Extract the deposit point coordinates and the distance from deposit points to granites
# and save them to a CSV file
result_deposit_granite = deposit_points_join[['geometry', 'granite_distance']]
result_deposit_granite.to_csv('deposit_granite_distances.csv', index=False)

In [ ]:
# Read the REE deposit data file
# Transform the coordinate reference system
deposit_coords = pd.read_csv(deposit_coords_file, index_col=False)
deposit_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(deposit_coords['X'], deposit_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the deposit points with the strata grid points in the frame
deposit_points_join = gpd.sjoin_nearest(deposit_points, points_within_frame_strata, how='left', distance_col='distance_to_grid')
deposit_points_join['strata_distance'] = deposit_points_join['distance']

# Extract the deposit point coordinates and the distance from deposit points to strataes
# and save them to a CSV file
result_deposit_strata = deposit_points_join[['geometry', 'strata_distance']]
result_deposit_strata.to_csv('deposit_strata_distances.csv', index=False)

#### Create the Data File of Mineral Occurrences



In [ ]:
# Define the path of the data file for REE deposits
deposit_training_data_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/deposit_training_data_file_7.11_d0.0115_all_109.csv'

# If file exists
if os.path.isfile(deposit_training_data_file):
    print('The training data file (deposits) already exists.')
    deposit_training_data = pd.read_csv(deposit_training_data_file, index_col=False)
    deposit_training_data_columns = deposit_training_data.columns.tolist()
    deposit_num_data_columns = [] #Stores numeric feature column names

    # Treat all columns as numerical features (except label columns)
    for column in deposit_training_data_columns:
        if column !='label':
            deposit_num_data_columns.append(column)

    # Extract numerical feature data
    deposit_num_data = deposit_training_data[deposit_num_data_columns]

# If file does not exist
else:
    deposit_training_data = pd.concat([
        deposit_coords,
        deposit_grid_data,
        deposit_elev_data

    ],
        axis=1)

    # Remove the samples with missing values
    deposit_training_data = deposit_training_data.dropna()

    # Extract numerical feature data
    deposit_num_data = deposit_training_data[deposit_training_data.columns[3:]]


    unique_columns_num = []
    # Remove columns (features) with a unique value from the list of numerical features
    for i in range(deposit_num_data.shape[1]):
        if len(deposit_num_data.iloc[:, i].round(4).unique()) == 1:
            unique_columns_num.append(deposit_num_data.columns[i])
    deposit_num_data.drop(unique_columns_num, axis=1, inplace=True)

    # Remain other numerical feature data as final feature data
    deposit_features = deposit_num_data

    # Extract label data
    deposit_labels = deposit_training_data[deposit_training_data.columns[2]].reset_index(drop=True)

    # Concatenate label and feature data and save them to a CSV file
    deposit_training_data = pd.concat([deposit_labels, deposit_features], axis=1).reset_index(drop=True)
    deposit_training_data.to_csv(deposit_training_data_file, index=False)
    print(f'The training data file (deposits) has been saved to {deposit_training_data_file}.')

In [ ]:
# Combine the values of three geological factors corresponding to deposits into one dataset

train_deposit_geo = pd.concat([result_deposit_fault.iloc[:, 1:], result_deposit_granite.iloc[:, 1:], result_deposit_strata.iloc[:, 1:]], axis=1)
train_deposit_geo.to_csv('train_deposit_geo.csv', index=False)


### Random (Unlabelled) Samples (If unlabeled sample data is not available, this code can be used to generate)

In [ ]:
# Generate and extract coordinates of a certain number of random samples in the study area
def get_unlab_samples(polygon, num_features):
    bounds = polygon.bounds
    rand_x = np.random.uniform(low=extent[0], high=extent[1], size=num_features*10)
    rand_y = np.random.uniform(low=extent[2], high=extent[3], size=num_features*10)
    unlab_x = []
    unlab_y = []

    # Check whether the generated random points are inside the study area
    for x, y in zip(rand_x, rand_y):
        if len(unlab_x) == num_features*5: # If the required number of samples is reached, stop the cycle
            break
        p = Point((x, y))
        if p.within(polygon.geometry[0]): # If the point is inside the study area, it is added to the list
            unlab_x.append(x)
            unlab_y.append(y)

    return unlab_x, unlab_y

num_features = deposit_training_data.shape[1] - 1

# Define the path to the CSV file to save the unlabeled sample coordinates
unlab_coords_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/unlabel_data/random_point_train_0p0005_863.csv'

# If file exists
if os.path.isfile(unlab_coords_file):
    print('The coordinates of unlabelled samples already exist.')
    # Read coordinate data
    unlab_coords = pd.read_csv(unlab_coords_file, index_col=False)
    unlab_x = pd.Series.tolist(unlab_coords['X'])
    unlab_y = pd.Series.tolist(unlab_coords['Y'])

# If file does not exist
else:
    # Generate and extract coordinates of unlabelled samples
    unlab_x, unlab_y = get_unlab_samples(frame, num_features)
    unlab_coords = pd.DataFrame(unlab_x, columns=['X'])
    unlab_coords['Y'] = unlab_y
    # Add  the label column
    unlab_label = [0]*len(unlab_x)
    unlab_coords['label'] = unlab_label
    # Save the coordinates and labels of unlabelled samples to a CSV file
    unlab_coords.to_csv(unlab_coords_file, index=False)
    print(f'The coordinates of unlabelled samples have been saved to {unlab_coords_file}.')


# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
ax.scatter(unlab_x, unlab_y, color='blue', edgecolors='black', s=10)
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.set_title('Unlabelled Samples')
plt.show()

#### Create the Unlabelled Data File of Random Samples

In [ ]:
# Export a file with unlabeled sample grid data
unlab_grid_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/training_data_unlab_grids_7.11_d0.0115_all_lowdensity_109.csv'

# If file exists
if os.path.isfile(unlab_grid_file):
    print('The grid dataset (unlabelled samples) already exists.')
    # Read grid data
    unlab_grid_data = pd.read_csv(unlab_grid_file, index_col=False)

# If file does not exist
else:
    # Calculate the grid features
    unlab_grid_data = get_grid_data(unlab_x, unlab_y, grid_filenames)
    # Retain only the same features as in the dataset of REE deposits
    unlab_grid_data = unlab_grid_data[unlab_grid_data.columns.intersection(deposit_grid_data.columns)]
    unlab_grid_data.to_csv(unlab_grid_file, index=False)
    print(f'The grid dataset (unlabelled samples) has been saved to {unlab_grid_file}.')

# Export a file with unlabeled sample elevation data
unlab_elev_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/training_data_unlab_elevation_7.11_d0.0115_all_lowdensity_109.csv'

# If file exists
if os.path.isfile(unlab_elev_file):
    print('The elevation dataset (unlabelled samples) already exists.')
    # Read elevation data
    unlab_elev_data = pd.read_csv(unlab_elev_file, index_col=False)

# If file does not exist
else:
    # Calculate the elevation features
    unlab_elev_data = get_gradient_data(unlab_x, unlab_y, [dem_file])
    # Retain only the same features as in the dataset of REE deposits
    unlab_elev_data = unlab_elev_data[unlab_elev_data.columns.intersection(deposit_elev_data.columns)]
    unlab_elev_data.to_csv(unlab_elev_file, index=False)
    print(f'The elevation dataset (unlabelled samples) has been saved to {unlab_elev_file}.')

### Create the Data File (including geophysical and geological features) of All Samples

In [ ]:
# Export a data file with unlabeled samples
unlab_training_data_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/training_data_unlab_7.11_d0.0115_all_random_0.65_all_109.csv'

# If file exists
if os.path.isfile(unlab_training_data_file):
    print('The training data file (unlabelled samples) already exists.')
    # Read data
    unlab_training_data = pd.read_csv(unlab_training_data_file, index_col=False)

# If file does not exist
else:
    #Combine coordinate, grid, and elevation data into the dataset
    unlab_training_data = pd.concat([
        unlab_coords,
        unlab_grid_data,
        unlab_elev_data],

        axis=1
    )

    # Remove missing values
    unlab_training_data = unlab_training_data.dropna()

    # Retain only the same features as in the dataset of REE deposits
    unlab_training_data = unlab_training_data[unlab_training_data.columns.intersection(deposit_training_data.columns)]
    unlab_training_data.to_csv(unlab_training_data_file, index=False)
    print(f'The training data file (unlabelled samples) has been saved to {unlab_training_data_file}.')

# Export a feature file for all samples, both labeled and unlabeled
Xy_train_original_df_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_original_7.11_d0.0115_all_random_0.65_all_109.csv'

# If file exists
if os.path.isfile(Xy_train_original_df_file):
    # Read data
    Xy_train_original_df = pd.read_csv(Xy_train_original_df_file, index_col=False)
    print('Features file already exists!')
    # Load the saved standardized model
    with open('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/st_scaler_7.11_d0.0115_all_random_0.65_all.pkl', 'rb') as f:
        st_scaler = pickle.load(f)

# If file does not exist
else:
    # Set the unlabeled sample label to 0 and merge it with the labeled sample
    deposit_labels = deposit_training_data['label']
    unlab_training_data['label'] = 0
    unlab_labels = unlab_training_data['label']
    labels = pd.concat([deposit_labels, unlab_labels]).reset_index(drop=True)
    training_data_original = pd.concat([deposit_training_data, unlab_training_data]).reset_index(drop=True)

    # Extract numerical feature data
    training_data_num = training_data_original[deposit_num_data.columns]


    # Drop highly correlated features
    # Create a correlation matrix
    corr_matrix = training_data_num.corr(method='spearman').abs()
    # Select the upper triangle of the correlation matrix
    corr_upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    # Find features with the correlation greater than 0.65
    corr_drop = [column for column in corr_upper.columns if any(corr_upper[column] > 0.65)]
    print('List of the features removed due to high correlation with other features:', corr_drop)
    # Drop features
    training_data_num_purged = training_data_num.drop(corr_drop, axis=1)

    # Combian numberical feature with label
    features_labels_encoded = pd.concat([training_data_num_purged, labels], axis=1).reset_index(drop=True)
    features_labels_list = features_labels_encoded.columns.tolist()
    features_list = features_labels_list.copy()
    features_list.remove('label')

    # Split the data into labelled and unlabelled samples
    deposit_data = features_labels_encoded[features_labels_encoded['label']==1]
    unlab_data = features_labels_encoded[features_labels_encoded['label']==0]

    deposit_features = deposit_data[deposit_data.columns[:-1]]
    unlab_features = unlab_data[unlab_data.columns[:-1]]

    deposit_labels = deposit_data[deposit_data.columns[-1]]
    unlab_labels = unlab_data[unlab_data.columns[-1]]

    # Merge all features and labels of samples
    X_all = np.vstack((deposit_features, unlab_features))
    y_all = np.vstack((deposit_labels.values.reshape(-1, 1), unlab_labels.values.reshape(-1, 1)))
    Xy_all_original = np.hstack((X_all, y_all))
    Xy_all_original_df = pd.DataFrame(Xy_all_original, columns=features_labels_list)
    Xy_all_original_df.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_original_7.11_d0.0115_all_random_0.65_all_109.csv', index=False)

    # Normalized numberical features
    X_train_num = Xy_all_original_df[training_data_num_purged.columns]
    st_scaler = MinMaxScaler()
    X_train_num = st_scaler.fit_transform(X_train_num)
    X_train_num = pd.DataFrame(X_train_num, columns=training_data_num_purged.columns)
    Xy_all = pd.concat([X_train_num, Xy_all_original_df['label']], axis=1).reset_index(drop=True)
    Xy_all.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_minmax_7.11_d0.0115_all_random_0.65_all_109.csv', index=False)

    # Save the standard scaler model
    with open('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/st_scaler_7.11_d0.0115_all_random_0.65_all.pkl', 'wb') as f:
        pickle.dump(st_scaler, f)



In [ ]:
# Read the unlabelled data file
# Set the coordinate reference system
unlab_coords = pd.read_csv(unlab_coords_file, index_col=False)
unlab_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(unlab_coords['X'], unlab_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the unlablled sample points with the fault grid points in the frame
unlab_points_join = gpd.sjoin_nearest(unlab_points, points_within_frame_faults, how='left', distance_col='distance_to_grid')
unlab_points_join['fault_distance'] = unlab_points_join['distance']

# Extract the unlabelled sample coordinates and the distance from unlabelled sample points to faults
# and save them to a CSV file
result_unlab_fault = unlab_points_join[['geometry', 'fault_distance']]
result_unlab_fault.to_csv('unlab_fault_distances.csv', index=False)

In [ ]:
# Read the unlabelled data file
# Set the coordinate reference system
unlab_coords = pd.read_csv(unlab_coords_file, index_col=False)
unlab_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(unlab_coords['X'], unlab_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the unlabelled sample points with the granite grid points in the frame
unlab_points_join = gpd.sjoin_nearest(unlab_points, points_within_frame_granite, how='left', distance_col='distance_to_grid')
unlab_points_join['granite_distance'] = unlab_points_join['distance']

# Extract the unlabelled sample coordinates and the distance from unlabelled sample points to granites
# and save them to a CSV file
result_unlab_granite = unlab_points_join[['geometry', 'granite_distance']]
result_unlab_granite.to_csv('unlab_granite_distances.csv', index=False)

In [ ]:
# Read the unlabelled data file
# Set the coordinate reference system
unlab_coords = pd.read_csv(unlab_coords_file, index_col=False)
unlab_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(unlab_coords['X'], unlab_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the unlabelled sample points with the strata grid points in the frame
unlab_points_join = gpd.sjoin_nearest(unlab_points, points_within_frame_strata, how='left', distance_col='distance_to_grid')
unlab_points_join['strata_distance'] = unlab_points_join['distance']

# Extract the unlabelled sample coordinates and the distance from unlabelled sample points to strataes
# and save them to a CSV file
result_unlab_strata = unlab_points_join[['geometry', 'strata_distance']]
result_unlab_strata.to_csv('unlab_strata_distances.csv', index=False)

In [ ]:
# Combine the values of three geological factors corresponding to unlablled sample points into one dataset
train_unlab_geo = pd.concat([result_unlab_fault.iloc[:, 1:], result_unlab_granite.iloc[:, 1:], result_unlab_strata.iloc[:, 1:]], axis=1)
train_unlab_geo.to_csv('unlab_train_geo.csv', index=False)

## Generate a feature data file for all samples: Integrating Geochemical, Geological, and Geophysical features

In [ ]:
# Combine the geological feature data of labeled and unlabeled samples by row
train_result_geo = pd.concat([train_deposit_geo, train_unlab_geo], axis=0)
train_result_geo = train_result_geo.reset_index(drop=True)
train_result_geo.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/train_result_geo_grid.csv', index=False)

In [ ]:
train_result_geo=pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/train_result_geo_grid.csv')

In [ ]:
# Combine the geophsical feature of both labeled and unlabeled samples with geological feature
all_grid=pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_minmax_7.11_d0.0115_all_random_0.65_all_109.csv')
result_train_grid_geo = pd.concat([train_result_geo, all_grid], axis=1)
result_train_grid_geo.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/result_train_grid_geo_109.csv', index=False)

In [ ]:
# If geochemical data is available, it can be imported into the experiment.
# In this experiment, we use R programming language to write code to process and analyze the original geochemical data.
# The processed geochemical data were imported into the experimental environment.
geochemical_data=pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/unlabel_data/geochemical_ilrrpca_back_pc7_zy_randomal5_814_2.csv')
# Normalized the geochemical data
scaler = MinMaxScaler()
geochemical_data_scaled = scaler.fit_transform(geochemical_data.iloc[:,2:])
geochemical_data_scaled_df = pd.DataFrame(geochemical_data_scaled, columns=geochemical_data.columns[2:])
# Insert the 'X' and 'Y' coordinates into the normalized geochemical data, keeping the original order.
xy_columns = geochemical_data[['X', 'Y']]
geochemical_data_scaled_df.insert(loc=0, column='X', value=xy_columns['X'])
geochemical_data_scaled_df.insert(loc=1, column='Y', value=xy_columns['Y'])
geochemical_data_scaled_df = pd.DataFrame(data = geochemical_data_scaled_df, columns=geochemical_data.columns[:])


In [ ]:
# Combine the geophsical, geological and geochemical features of labeled and unlabeled samples
concatenated_data_train = pd.concat([geochemical_data_scaled_df, result_train_grid_geo], axis=1)
print(concatenated_data_train.shape)
concatenated_data_train.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/ALL_phyd0.0011t0.65_cheilrrpcabackd0.01_geo3_traindata_random_109.csv',index=False)


# Creating Negative Sample Set: Based on Distance Threshold

In [ ]:
# Read the location data of known REE deposits
min_occ = gpd.read_file('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/GIS/ree.shp')
# Read feature data for all sample points (including geophysical, geological and geochemical data)
all_points = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/ALL_phyd0.0011t0.65_cheilrrpcabackd0.01_geo3_traindata_random_109.csv')

# Define a distance threshold in meters
threshold = 5000

# Screen negative sample
negative_samples = []
for _, point in all_points.iterrows():
    exceed_threshold = True
    for _, known_point in min_occ.iterrows():
        # Calculate the distance between a sample point and a known REE depsoit
        if distance.distance((point['Y'], point['X']), (known_point.geometry.y, known_point.geometry.x)).meters <= threshold:
            exceed_threshold = False # # If the distance is less than or equal to the threshold, the sample point is not a negative sample
            break
    # If the distance from all known REE deposits exceeds the threshold, the sample point is added to the list of negative samples
    if exceed_threshold:
        negative_samples.append(point)


negative_samples_df = pd.DataFrame(negative_samples)
# Add a label column for negative samples
negative_samples_df['label'] = 0
# Extract all positive samples
positive_samples_df = all_points[all_points['label'] == 1]

# Combine the positive and negative samples into a single dataset
combined_dataset = pd.concat([positive_samples_df, negative_samples_df], ignore_index=True)
combined_dataset.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/combined_dataset_109.csv', index=False)


In [ ]:
# Read the coordinates of the positive and negative samples
PaN_coords_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/combined_dataset_109.csv'
PaN_coords = pd.read_csv(PaN_coords_file, index_col=False)
PaN_x = pd.Series.tolist(PaN_coords['X'])
PaN_y = pd.Series.tolist(PaN_coords['Y'])



# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
ax.scatter(PaN_x, PaN_y, color='blue', edgecolors='black', s=10)
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.set_title('Negative Samples and 7 Positive Samples')
plt.show()

## Splitting Training and Testing Sets from Negative and Positive Samples

In [ ]:
def dataLoading(path):
    # loading data
    df = pd.read_csv(path)

    labels = df['label']

    x_df = df.drop(['X','Y','label'], axis=1)

    x = x_df.values
    print("Data shape: (%d, %d)" % x.shape)

    return x, labels

# Use the dataLoading function to load the data and get features and labels
x, labels = dataLoading('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/combined_dataset_109.csv')
# Divided into training sets and test sets
x_train, x_test, y_train, y_test = train_test_split(x, labels, test_size=0.3,random_state=56,stratify=labels)
print(x_train.shape)
print(x_test.shape)

In [ ]:
# Count the number of positive and negative samples
unique_labels, counts = np.unique(y_train, return_counts=True)
label_counts = pd.Series(counts, index=unique_labels)
print(label_counts)

### Create the Train Model


In [ ]:
# Set random seed
global_seed = 42
random.seed(global_seed)
np.random.seed(global_seed)
tf.random.set_seed(global_seed)


# Encapsulate the DevNet model
class DevNetWrapper:
    def __init__(self, input_shape, batch_size, nb_batch, epochs):
        self.input_shape = input_shape
        self.batch_size = batch_size
        self.nb_batch = nb_batch
        self.epochs = epochs
        self.model_ = None
        self.classes_ = None

    # Train the model
    def fit(self, X, y):
        self.model_ = deviation_network(self.input_shape)
        self.model_.fit(
            batch_generator_sup(X, np.where(y == 1)[0], np.where(y == 0)[0],
                                self.batch_size, self.nb_batch, np.random.RandomState(42)),
            steps_per_epoch=self.nb_batch,
            epochs=self.epochs)
        return self

    # Predict the output of the model (scores)
    def predict(self, X):
        if self.classes_ is None:
            self.classes_ = [0, 1]
        if self.model_ is None:
            raise ValueError("Model not fitted yet.")
        return self.model_.predict(X)

    # Predict the output of the model (probability)
    def predict_proba(self, X):
        if self.classes_ is None:
             self.classes_ = [0, 1]
        scores = self.predict(X)
        min_score = np.min(scores)
        max_score = np.max(scores)
        normalized_scores = (scores - min_score) / (max_score - min_score)
        probs = 1 / (1 + np.exp(-normalized_scores))
        probs = np.column_stack((1 - probs, probs))
        return probs

    # Get the parameters of the model
    def get_params(self, deep=True):
        return {
            "input_shape": self.input_shape,
            "batch_size": self.batch_size,
            "nb_batch": self.nb_batch,
            "epochs": self.epochs,
        }

    # Set the parameters of the model
    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

# Define the structure of DevNet
def dev_network_d(input_shape):
    x_input = Input(shape=input_shape)
    intermediate = Dense(24, activation='relu', kernel_regularizer=regularizers.l2(0.0005), name='hl1')(x_input)
    intermediate = Dense(12, activation='relu', kernel_regularizer=regularizers.l2(0.0007), name='hl2')(intermediate)
    #intermediate = Dense(5, activation='linear', name='hl3')(intermediate)
    intermediate = Dense(1, activation='linear', name='score')(intermediate)
    return Model(x_input, intermediate)

# Define the reference score
ref = K.variable(np.random.normal(loc=0., scale=1.0, size=50000), dtype='float32')
# Defined the loss function
def deviation_loss(y_true, y_pred):
    confidence_margin = 20
    dev = (y_pred - K.mean(ref)) / K.std(ref)
    inlier_loss = K.abs(dev)
    outlier_loss = K.abs(K.maximum(confidence_margin - dev, 0.))
    y_true = K.cast(y_true, dtype='float32')  # Cast y_true to float32
    return K.mean((1 - y_true) * inlier_loss + y_true * outlier_loss)

# Define the function to create the DevNet
def deviation_network(input_shape):
    model = dev_network_d(input_shape)
    nadam = Nadam(learning_rate=0.005, beta_1=0.9, beta_2=0.999)
    model.compile(loss=deviation_loss, optimizer=nadam)
    return model

# Definition the generator
def batch_generator_sup(x, outlier_indices, inlier_indices, batch_size, nb_batch, rng):
    rng = np.random.RandomState(rng.randint(MAX_INT, size=1))
    counter = 0
    while 1:
        ref, training_labels = input_batch_generation_sup(x, outlier_indices, inlier_indices, batch_size, rng)
        counter += 1
        yield (ref, training_labels)
        if (counter > nb_batch):
            counter = 0

# Define a function to generate a single batch of data
def input_batch_generation_sup(xx_train, outlier_indices, inlier_indices, batch_size, rng):
    dim = xx_train.shape[1]
    ref = np.empty((batch_size, dim))
    training_labels = []
    n_inliers = len(inlier_indices)
    n_outliers = len(outlier_indices)
    for i in range(batch_size):
        if (i % 2 == 0):
            sid = rng.choice(n_inliers, 1)
            ref[i] = xx_train[inlier_indices[sid]]
            training_labels += [0]
        else:
            sid = rng.choice(n_outliers, 1)
            ref[i] = xx_train[outlier_indices[sid]]
            training_labels += [1]
    return np.array(ref), np.array(training_labels)

# Function to calculate Youden's J at a given threshold
def youdens_j(true_labels, pred_scores, threshold):
    # Convert probabilities to binary predictions based on threshold
    pred_labels = (pred_scores >= threshold).astype(int)

    # Compute confusion matrix
    tn, fp, fn, tp = confusion_matrix(true_labels, pred_labels).ravel()

    # Calculate sensitivity (recall) and specificity
    sensitivity = tp / (tp + fn)  # True Positive Rate
    specificity = tn / (tn + fp)  # True Negative Rate

    # Youden's J statistic
    j_statistic = sensitivity + specificity - 1
    return j_statistic

# Function to find the optimal threshold dynamically
def find_best_threshold(y_true, y_scores, metric='f1'):
    best_threshold = 0.0
    best_score = -1.0
    thresholds = np.arange(0.0, 1.0, 0.01)  # Try thresholds from 0 to 1 with step of 0.01

    for threshold in thresholds:
        # Binarize predictions using the current threshold
        y_pred_binary = (y_scores >= threshold).astype(int)

        if metric == 'f1':
            # Calculate F1-score
            score = f1_score(y_true, y_pred_binary)
        elif metric == 'youden':
            # Calculate Youden's J statistic
            score = youdens_j(y_true, y_scores, threshold)
        else:
            raise ValueError("Unsupported metric. Use 'f1' or 'youden'.")

        # Update best threshold if the score is higher
        if score > best_score:
            best_score = score
            best_threshold = threshold

    return best_threshold, best_score

def evaluate_model_at_thresholds(y_true, y_pred_scores, thresholds):

    if not isinstance(thresholds, (list, np.ndarray)):
        thresholds = [thresholds]

    for threshold in thresholds:
        # Binarize predictions using the threshold
        y_pred_binary = (y_pred_scores > threshold).astype(int)

        # Calculate Accuracy
        accuracy = accuracy_score(y_true, y_pred_binary)

        # Calculate MCC
        mcc = matthews_corrcoef(y_true, y_pred_binary)

        # Calculate Cohen's Kappa
        kappa = cohen_kappa_score(y_true, y_pred_binary)

        print(f"Threshold: {threshold:.2f} -> Accuracy: {accuracy:.4f}, MCC: {mcc:.4f}, Cohen's Kappa: {kappa:.4f}")

# Visualiz weight and bias
def plot_weights_and_biases(model):
    for layer_idx, layer in enumerate(model.layers):
        if len(layer.get_weights()) > 0:
            weights, biases = layer.get_weights()

            print(f"\nLayer {layer_idx + 1} - {layer.name}")

            plt.figure(figsize=(12, 4))

            print(f"Weight shape: {weights.shape}")

            neurons_in_current_layer = weights.shape[1]
            neurons_in_previous_layer = weights.shape[0]

            plt.subplot(1, 2, 1)
            plt.title(f"Weights of layer {layer_idx + 1}")
            plt.imshow(weights, aspect='auto', cmap='viridis',
                       extent=[0, neurons_in_current_layer, 0, neurons_in_previous_layer])
            plt.colorbar()
            plt.xlabel('Neurons in current layer')
            plt.ylabel('Neurons in previous layer')

            print(f"Biases shape: {biases.shape}")
            plt.subplot(1, 2, 2)
            plt.title(f"Biases of layer {layer_idx + 1}")
            plt.imshow(biases.reshape(1, -1), aspect='auto', cmap='viridis',
                       extent=[0, neurons_in_current_layer, 0, 1])
            plt.colorbar()
            plt.xlabel('Neurons in current layer')
            plt.ylabel('Biases')

            plt.show()




MAX_INT = np.iinfo(np.int32).max

# Main function
if __name__ == '__main__':

    xx_train=x_train
    yy_train=y_train

    input_shape = xx_train.shape[1:]
    devnet = DevNetWrapper(input_shape, batch_size=128, nb_batch=5, epochs=500)
    devnet.fit(xx_train, yy_train)


    y_test_pred = devnet.predict(x_test)
    np.savetxt('y_test_pred.csv',y_test_pred)
    y_test_scores = devnet.predict_proba(x_test)[:, 1]
    np.savetxt('y_test_scores.csv',y_test_scores)


    auc_roc_test = roc_auc_score(y_test, y_test_scores)
    print(f"AUC-ROC: {auc_roc_test:.4f}")

    # Find the best threshold based on F1-score
    best_threshold_f1, best_f1_score = find_best_threshold(y_test, y_test_scores, metric='f1')
    print(f"Best Threshold (F1): {best_threshold_f1:.2f}, Best F1-Score: {best_f1_score:.4f}")

    # Find the best threshold based on Youden's J
    best_threshold_youden, best_youden_score = find_best_threshold(y_test, y_test_scores, metric='youden')
    print(f"Best Threshold (Youden's J): {best_threshold_youden:.2f}, Best Youden's J: {best_youden_score:.4f}")

    # Binarize predictions using the best threshold based on F1-score
    evaluate_model_at_thresholds(y_test, y_test_scores, best_threshold_f1)

    # Combine the training and testing datasets
    X_combined = np.vstack((x_train, x_test))  # Stack the input data
    y_combined = np.hstack((y_train, y_test))  # Stack the labels

    y_combined_pred = devnet.predict(X_combined)
    np.savetxt('y_combined_pred.csv',y_combined_pred)
    y_combined_scores = devnet.predict_proba(X_combined)[:, 1]
    np.savetxt('y_combined_scores.csv',y_combined_scores)

    auc_roc_test_combined = roc_auc_score(y_combined, y_combined_scores)
    auc_pr_test_combined = average_precision_score(y_combined, y_combined_scores)
    print(f"AUC-ROC: {auc_roc_test_combined:.4f}, AUC-PR: {auc_pr_test_combined:.4f}")


    # Find the best threshold based on F1-score
    best_threshold_f1_combined, best_f1_score_combined = find_best_threshold(y_combined, y_combined_scores, metric='f1')
    print(f"Best Threshold (F1)_combined: {best_threshold_f1_combined:.2f}, Best F1-Score_combined: {best_f1_score_combined:.4f}")

    # Find the best threshold based on Youden's J
    best_threshold_youden_combined, best_youden_score_combined = find_best_threshold(y_combined, y_combined_scores, metric='youden')
    print(f"Best Threshold (Youden's J)_combined: {best_threshold_youden_combined:.2f}, Best Youden's J_combined: {best_youden_score_combined:.4f}")

    # Binarize predictions using the best threshold based on F1-score
    evaluate_model_at_thresholds(y_combined, y_combined_scores, best_threshold_f1_combined)

    # Save the trained model
    devnet.model_.save("only_devnet_model.h5")

    # Visualiz weight and bias
    print("Visualizing model's weights and biases...")
    plot_weights_and_biases(devnet.model_)



## Creating target dataset

In [ ]:
# Read the coordinates of the target samples
target_coords_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/unlabel_data/regular_data_coord.csv'
target_coords = pd.read_csv(target_coords_file, index_col=False)
target_x = pd.Series.tolist(target_coords['X'])
target_y = pd.Series.tolist(target_coords['Y'])

# Create a drawing object and display it
fig, ax = plt.subplots(figsize=(15, 15))
ax.scatter(target_x, target_y, color='blue', edgecolors='black', s=10)
frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.set_title('Target Samples')
plt.show()

In [ ]:
# Export the grid data of target samples
target_grid_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/training_data_unlab_grids_7.11_d0.0115_all_pred_109.csv'

# Check if the grid dataset already exists
# If it exists, load it from the CSV file
if os.path.isfile(target_grid_file):
    print('The grid dataset (target samples) already exists.')
    target_grid_data = pd.read_csv(target_grid_file, index_col=False)

# If it does not exist
else:
    # Get the grid data of target samples
    target_grid_data = get_grid_data(target_x, target_y, grid_filenames)
    # Retain only the same features as in the dataset of REE deposits
    target_grid_data = target_grid_data[target_grid_data.columns.intersection(deposit_grid_data.columns)]
    target_grid_data.to_csv(target_grid_file, index=False)
    print(f'The grid dataset (target samples) has been saved to {target_grid_file}.')

# Export the elevation data of target samples
target_elev_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/training_data_unlab_elevation_7.11_d0.0115_all_pred_109.csv'

# Check if the elevation dataset already exists
# If it exists, load it from the CSV file
if os.path.isfile(target_elev_file):
    print('The elevation dataset (target samples) already exists.')
    target_elev_data = pd.read_csv(target_elev_file, index_col=False)

# If it does not exist
else:
    # Get the elevation data of target samples
    target_elev_data = get_gradient_data(target_x, target_y, [dem_file])
    # Retain only the same features as in the dataset of REE deposits
    target_elev_data = target_elev_data[target_elev_data.columns.intersection(deposit_elev_data.columns)]
    target_elev_data.to_csv(target_elev_file, index=False)
    print(f'The elevation dataset (target samples) has been saved to {target_elev_file}.')

In [ ]:
# Export a data file with target samples
target_training_data_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/training_data_unlab_7.11_d0.0115_all_regular_0.65_pred_109.csv'

# Check if the training data file already exists
# If it exists, load it from the CSV file
if os.path.isfile(target_training_data_file):
    print('The training data file (unlabelled samples) already exists.')
    target_training_data = pd.read_csv(target_training_data_file, index_col=False)

# If it does not exist
else:
    # Combine the coordinate, grid and elevation data of target samples into a single dataset
    target_training_data = pd.concat([
        target_coords,
        target_grid_data,
        target_elev_data],

        axis=1
    )

    # Remove missing values
    target_training_data = target_training_data.dropna()
    # Retain only the same features as in the dataset of REE deposits
    deposit_training_feature=deposit_training_data.drop(['label'], axis=1)
    target_training_data = target_training_data[target_training_data.columns.intersection(deposit_training_feature.columns)]
    target_training_data.to_csv(target_training_data_file, index=False)
    print(f'The data file (target samples) has been saved to {target_training_data_file}.')

# Export a feature file for all samples
Xy_target_original_df_file = '/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_original_7.11_d0.0115_all_regular_0.65_pred_109.csv'

# Check if the features file already exists
# If it exists, load it from the CSV file
if os.path.isfile(Xy_target_original_df_file):
    Xy_target_original_df = pd.read_csv(Xy_target_original_df_file, index_col=False)
    print('Features file already exists!')

    # Load the saved standardized model
    with open('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/st_scaler_7.11_d0.0115_all_regular_0.65_pred.pkl', 'rb') as f:
        st_scaler = pickle.load(f)

# If it does not exist,
else:
    # Combine known REE deposit data with target sample data to generate a complete dataset
    target_training_data_original = pd.concat([deposit_training_feature, target_training_data]).reset_index(drop=True)
    Xy_target_original_df = target_training_data_original
    Xy_target_original_df.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_original_7.11_d0.0115_all_regular_0.65_pred_109.csv', index=False)

    # Normalized merge data
    target_X_train_num = Xy_target_original_df
    st_scaler = MinMaxScaler()
    target_X_train_num = st_scaler.fit_transform(target_X_train_num)
    target_X_train_num = pd.DataFrame(target_X_train_num, columns=target_training_data_original.columns)
    Xy_all_target = target_X_train_num
    Xy_all_target.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_minmax_7.11_d0.0115_all_regular_0.65_pred_109.csv', index=False)


    # Save the standard scaler model
    with open('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/st_scaler_7.11_d0.0115_all_regular_0.65_pred.pkl', 'wb') as f:
        pickle.dump(st_scaler, f)



In [ ]:
# Load the training data
geophysical_data_train=pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_minmax_7.11_d0.0115_all_random_0.65_all_109.csv')
print(geophysical_data_train.shape)
# Get a list of columns from the training data
columns_to_keep = geophysical_data_train.columns.tolist()
geophysical_data_pred = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/Xy_all_minmax_7.11_d0.0115_all_regular_0.65_pred_109.csv')
# Filter columns in the prediction data
geophysical_data_pred_filtered = geophysical_data_pred[geophysical_data_pred.columns.intersection(columns_to_keep)]
print(geophysical_data_pred_filtered.shape)
geophysical_data_pred_filtered.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/geophysical_data_pred_filtered_109.csv', index=False)

In [ ]:
# If geochemical data is available, it can be imported into the experiment.
# In this experiment, we use R programming language to write code to process and analyze the original geochemical data.
# The processed geochemical data were imported into the experimental environment.
geochemical_data_target=pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/unlabel_data/ilrrpca_back_regular_pred_0p015_711_all_PC.csv')
geochemical_data_feature_target=geochemical_data_target.iloc[:,2:]
print(geochemical_data_feature_target.shape)
# Normalized the geochemical data
scaler = MinMaxScaler()
geochemical_data_target_scaled = scaler.fit_transform(geochemical_data_feature_target)
geochemical_data_target_scaled_df = pd.DataFrame(geochemical_data_target_scaled, columns=geochemical_data_feature_target.columns[:])


In [ ]:
# Read the target data
# Set the coordinate reference system
target_coords = pd.read_csv(target_coords_file, index_col=False)
target_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(target_coords['X'], target_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the target sample points with the fault grid points in the frame
target_points_join = gpd.sjoin_nearest(target_points, points_within_frame_faults, how='left', distance_col='distance_to_grid')
target_points_join['Archean_Early_Mesoproterozoic_Faults'] = target_points_join['distance']

# Extract the unlabelled sample coordinates and the distance from unlabelled sample points to strataes
# and save them to a CSV file
result_target_fault = target_points_join[['geometry', 'Archean_Early_Mesoproterozoic_Faults']]
result_target_fault.to_csv('target_fault_distances.csv', index=False)

In [ ]:
# Read the target data
# Set the coordinate reference system
target_coords = pd.read_csv(target_coords_file, index_col=False)
target_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(target_coords['X'], target_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the target sample points with the granite grid points in the frame
target_points_join = gpd.sjoin_nearest(target_points, points_within_frame_granite, how='left', distance_col='distance_to_grid')
target_points_join['Felsic_granite'] = target_points_join['distance']

# Extract the target sample coordinates and the distance from target sample points to strataes
# and save them to a CSV file
result_target_granite = target_points_join[['geometry', 'Felsic_granite']]
result_target_granite.to_csv('target_granite_distances.csv', index=False)

In [ ]:
# Read the target data
# Set the coordinate reference system
target_coords = pd.read_csv(target_coords_file, index_col=False)
target_points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(target_coords['X'], target_coords['Y']), crs='epsg:4283')

# Carry out the spatial connection
# Match the target sample points with the strata grid points in the frame
target_points_join = gpd.sjoin_nearest(target_points, points_within_frame_strata, how='left', distance_col='distance_to_grid')
target_points_join['Mesoproterozoic_strata'] = target_points_join['distance']

# Extract the target sample coordinates and the distance from target sample points to strataes
# and save them to a CSV file
result_target_strata = target_points_join[['geometry', 'Mesoproterozoic_strata']]
result_target_strata.to_csv('target_strata_distances.csv', index=False)

In [ ]:
# Combine three geophsical features of unlabeled target samples
target_unlab_geo = pd.concat([result_target_fault.iloc[:, 1:], result_target_granite.iloc[:, 1:], result_target_strata.iloc[:, 1:]], axis=1)
target_unlab_geo.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/pred_target_geo_109.csv', index=False)

In [ ]:
# Combine geophsical features of REE deposits and unlabeles target samples
df1_train_deposit_geo = train_deposit_geo
df2_target_unlab_geo = target_unlab_geo
df1_train_deposit_geo = df1_train_deposit_geo.rename(columns={'fault_distance': 'Archean_Early_Mesoproterozoic_Faults', 'granite_distance': 'Felsic_granite', 'strata_distance': 'Mesoproterozoic_strata'})
target_all_geo = pd.concat([df1_train_deposit_geo, df2_target_unlab_geo], ignore_index=True)
target_all_geo.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/target_all_geo_109.csv', index=False)

In [ ]:
# Read geophsical features of target samples
geophysical_data_target=pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/clip_dataset/outputs/geophysical_data_pred_filtered_109.csv')
geophysical_feature_target=geophysical_data_target#.iloc[:,0:-1]
#label_target=geophysical_data_target.iloc[:,-1]

In [ ]:
# Combine the geophsical, geological and geochemical features of REE deposits and target samples
concatenated_data_pred = pd.concat([geochemical_data_target_scaled_df,target_all_geo, geophysical_feature_target], axis=1)
concatenated_data_pred.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/ALL_phyd0.0011t0.65_cheilrrpcabackd0.01_geo3_preddata_regular_109.csv',index=False)

In [ ]:
print(concatenated_data_pred.shape)

## Creating prediction model

In [ ]:
# Read target dataset
x_pred = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/ALL_phyd0.0011t0.65_cheilrrpcabackd0.01_geo3_preddata_regular_109.csv')

# Predict Probabilities
y_pred = devnet.predict_proba(x_pred)[:, 1]
print(y_pred.shape)
np.savetxt('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/georesults/ALL_preddata_regular_result_proba_723_devnet_shijidistance_un30_r1_3ceng_nat500_cf72_pred_109.csv',y_pred)

## Interpreting model predictions

In [ ]:
# Insert the Scores into the Original target feature Data
Ysdata = x_pred
YS=np.array(Ysdata)
df = pd.DataFrame(data = y_pred, columns = ['scores'])
D=np.array(df)
Y=np.insert(YS,58,D.T,axis=1)
print(Y.shape)
np.savetxt('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/Ysaddscor.csv',Y)
Y

In [ ]:
# Sort the  samples according to the predicting probability value
preddata=pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/ALL_phyd0.0011t0.65_cheilrrpcabackd0.01_geo3_preddata_regular_109.csv')
Hostdata = np.loadtxt('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/Ysaddscor.csv', delimiter=' ')
Ho=np.array(Hostdata)
idx = Ho[:,-1].argsort()
new_idx = idx[::-1]
Ho = Ho[new_idx]
np.savetxt('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/sort_score.csv',Ho)
# Remove the Last Column (Predicted Scores)
deysx=np.delete(Ho,58, axis=1)
print(deysx.shape)
dex = pd.DataFrame(data = deysx, columns=preddata.columns[:] )
print(dex.shape)
dex.to_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/sort_score_input.csv',header=True, sep=' ')
Xhost = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/sort_score_input.csv',delimiter=' ')
XH_test = Xhost
print(XH_test.shape)
# Remove sequence number column
XH_test=XH_test.drop(XH_test.columns[0], axis=1)
XH_standard=XH_test
print(XH_standard)
np.savetxt('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/sortrec_err_XH.csv',XH_standard)

In [ ]:
# Load the JavaScript needed for SHAP plots to display interactively in the notebook.
shap.initjs()

In [ ]:
# Create a background dataset for SHAP (SHapley Additive exPlanations) using k-means clustering
background = shap.kmeans(x_train, 10)

In [ ]:
# KernelExplainer Initialization
e = shap.KernelExplainer(devnet.predict,background)
# Output expected Value
print(e.expected_value)

In [ ]:
# Use shap.kernelexplainer to explain the sorted target dataset and calculate the SHAP value
shap_values = e.shap_values(XH_standard)

In [ ]:
print(shap_values.shape)

In [ ]:
# Interpret a single sample (the seventh) and generate a waterfall map
sample_index = 7
shap.waterfall_plot(shap.Explanation(values=shap_values[sample_index, :,0],
                                     base_values=e.expected_value[0]
                                  ))

In [ ]:
# Interpret all samples
shap.summary_plot(shap_values[0:679,:,0], XH_standard)

In [ ]:
# Interpret all samples (Bar chart)
plt.figure()

shap.summary_plot(shap_values[0:679,:,0], XH_standard, plot_type="bar", show=False)

plt.gcf().set_size_inches(12, 8)

plt.savefig('barchart.png', bbox_inches='tight')

In [ ]:
# Interpret partial samples and generate decision graphs
shap.decision_plot(e.expected_value[0], shap_values[0:57,:,0], XH_standard)

## Prospectivity maps

In [ ]:
# Normalized predictive values
def min_max_norm(data):
    x_min = min(data)
    x_max = max(data)
    return [(x - x_min) / (x_max - x_min) for x in data]
predict_result=np.loadtxt('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/georesults/ALL_preddata_regular_result_proba_723_devnet_shijidistance_un30_r1_3ceng_nat500_cf72_pred_109.csv',delimiter=',')
normalized_predict_result = np.array(min_max_norm(predict_result))
point_nor_predict_mean = normalized_predict_result.mean()
point_nor_predict_var = normalized_predict_result.var()
print(predict_result)
print(normalized_predict_result)

In [ ]:
# Read the coordinates of the target data
sample_point_xy=np.loadtxt('/content/gdrive/MyDrive/Colab Notebooks/Au_MPM/DevNet/geodata/XY_all_pred_target.csv',delimiter=',',skiprows=1)
print(sample_point_xy)

# Merge the normalized prediction with the coordinates
predict_target=np.hstack((sample_point_xy, normalized_predict_result.reshape(-1, 1)))
column_names =['X','Y','Scores']
predict_target_result = np.zeros(predict_target.shape[0], dtype=[(name, 'f8') for name in column_names])
for i, name in enumerate(column_names):
    predict_target_result[name] = predict_target[:, i]
print(predict_target_result)

In [ ]:
# Extract the coordinates of the target data
plot_x = predict_target_result['X']
plot_y = predict_target_result['Y']

# Plot the probability map
fig, ax = plt.subplots(figsize=(15, 15))
cb = plt.scatter(plot_x, plot_y, 32., c=predict_target_result['Scores'], cmap='Spectral_r')

frame.plot(ax=ax, edgecolor='red', color='none', linewidth=2)
cx.add_basemap(ax, crs='EPSG:4283', source=cx.providers.Esri.WorldGrayCanvas)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.f'))
plt.yticks(rotation=90, va='center')
ax.xaxis.set_major_formatter(FormatStrFormatter('%.f'))

# Colorbar
divider = make_axes_locatable(ax)
cax = divider.new_vertical(size='5%', pad=0.5, pack_start=True)
fig.add_axes(cax)
plt.colorbar(cb, orientation='horizontal', label='Probability', cax=cax)
plt.show()

In [ ]:
# Extract the coordinates and scores of the target data
sample_x = predict_target_result['X']
sample_y = predict_target_result['Y']
scores = predict_target_result['Scores']

# Compute grid increment
x_range = sample_x.max() - sample_x.min()
y_range = sample_y.max() - sample_y.min()
x_inc = x_range / 100
y_inc = y_range / 100
region = [sample_x.min(), sample_x.max(), sample_y.min(), sample_y.max()]
spacing = f'{x_inc}/{y_inc}'

# Generate grid
grid = pygmt.surface(
    x=sample_x,
    y=sample_y,
    z=scores,
    region=region,
    spacing=spacing,
    tension=0.8
)
grid_data = grid.to_numpy()
grid_x = np.linspace(region[0], region[1], grid_data.shape[1])
grid_y = np.linspace(region[2], region[3], grid_data.shape[0])

# Plot the probability map
fig, ax = plt.subplots(figsize=(12, 10))
frame.boundary.plot(ax=ax, edgecolor='black', linewidth=2)
frame_mask = frame.geometry.unary_union
frame_gdf = gpd.GeoDataFrame(geometry=[frame_mask], crs='EPSG:4283')
frame_gdf = frame_gdf.to_crs(epsg=4283)
im = ax.imshow(grid_data, extent=region, origin='lower', cmap='viridis',
               norm=Normalize(vmin=scores.min(), vmax=scores.max()),
               alpha=0.7)
cbar = plt.colorbar(im, ax=ax, label='Scores')
ax.scatter(deposit_x, deposit_y, c='red', s=20, edgecolor='black')
ax.set_xlim(region[0], region[1])
ax.set_ylim(region[2], region[3])
frame.boundary.plot(ax=ax, edgecolor='black', linewidth=2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.title('Prospectivity Map of REE Mineralization')
plt.show()